In [0]:
STORAGE_ACCOUNT  = "storageaccountfull"
BRONZE_CONTAINER = "bronze"
BRONZE_PATH      = f"abfss://{BRONZE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

CATALOG          = "e-commerce"
SILVER_DB        = "silver"
TARGET_TABLE     = "fact_sales"
NATURAL_KEY      = "order_id"

In [0]:
import warnings
import math
warnings.filterwarnings("ignore", message=".*No Partition Defined for Window operation.*")

from pyspark.sql.functions import (
    col, trim, when, lit, year, expr,
    current_timestamp, coalesce,
    round as spark_round
)
from pyspark.sql.types import (
    IntegerType, DateType, DoubleType
)
from delta.tables import DeltaTable

spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"CREATE DATABASE IF NOT EXISTS {SILVER_DB}")
print("Imports ready | Database ensured")

In [0]:
df_bronze = (
    spark.read
    .format("parquet")
    .load(f"{BRONZE_PATH}/sales")
    .withColumn("source_file", col("_metadata.file_path"))
)

print(f"Bronze rows : {df_bronze.count():,}")
df_bronze.printSchema()
df_bronze.show(5, truncate=False)

In [0]:
total = df_bronze.count()

df_keyed = df_bronze.filter(
    col("order_id").isNotNull()    &
    col("product_id").isNotNull()  &
    col("store_id").isNotNull()    &
    col("customer_id").isNotNull() &
    col("order_date").isNotNull()
)

print(f"Null-key rows dropped : {total - df_keyed.count():,}")
print(f"Rows kept             : {df_keyed.count():,}")

In [0]:
df_clean = (
    df_keyed
    .withColumn("order_id",    trim(col("order_id")))
    .withColumn("product_id",  trim(col("product_id")))
    .withColumn("store_id",    trim(col("store_id")))
    .withColumn("customer_id", trim(col("customer_id")))
    .withColumn("order_date",
        coalesce(
            expr("try_to_date(order_date, 'yyyy-MM-dd')"),
            expr("try_to_date(order_date, 'dd-MM-yyyy')")
        )
    )
    .withColumn("quantity",    col("quantity").cast(IntegerType()))
    .withColumn("unit_price",  col("unit_price").cast(DoubleType()))
    .withColumn("discount",    col("discount").cast(DoubleType()))
    .withColumn("revenue",     col("revenue").cast(DoubleType()))
    .withColumn("cost",        col("cost").cast(DoubleType()))
    .withColumn("profit",      col("profit").cast(DoubleType()))
)

print("Types fixed")

In [0]:
df_validated = (
    df_clean
    .withColumn("is_valid",
        when(
            (col("quantity")   > 0)         &
            (col("unit_price") > 0)         &
            (col("discount").between(0, 1)) &
            (col("revenue")    >= 0),
            lit(1)
        ).otherwise(lit(0))
    )
)

print(f"Valid rows   : {df_validated.filter(col('is_valid') == 1).count():,}")
print(f"Invalid rows : {df_validated.filter(col('is_valid') == 0).count():,}  (kept, flagged)")

In [0]:
before     = df_validated.count()
df_deduped = df_validated.dropDuplicates([NATURAL_KEY])
print(f"Before : {before:,}  |  After : {df_deduped.count():,}")

In [0]:
dim_customers = (
    spark.table(f"{SILVER_DB}.dim_customers")
    .select("customer_id", "customer_sk")
)

dim_products = (
    spark.table(f"{SILVER_DB}.dim_products")
    .select("product_id", "product_sk")
)

dim_stores = (
    spark.table(f"{SILVER_DB}.dim_stores")
    .select("store_id", "store_sk")
)

dim_calendar = (
    spark.table(f"{SILVER_DB}.dim_calendar")
    .select(col("date").alias("order_date"), "date_sk")
)

print("Dim tables loaded:")
print(f"   dim_customers : {dim_customers.count():,} rows")
print(f"   dim_products  : {dim_products.count():,} rows")
print(f"   dim_stores    : {dim_stores.count():,} rows")
print(f"   dim_calendar  : {dim_calendar.count():,} rows")

In [0]:
df_fact = (
    df_deduped
    .join(dim_customers, on="customer_id", how="left")
    .join(dim_products,  on="product_id",  how="left")
    .join(dim_stores,    on="store_id",    how="left")
    .join(dim_calendar,  on="order_date",  how="left")
)

print(f"After dim joins : {df_fact.count():,} rows")

orphan_customers = df_fact.filter(col("customer_sk").isNull()).count()
orphan_products  = df_fact.filter(col("product_sk").isNull()).count()
orphan_stores    = df_fact.filter(col("store_sk").isNull()).count()
orphan_dates     = df_fact.filter(col("date_sk").isNull()).count()

print(f"\nFK Resolution Check")
print(f"   NULL customer_sk : {orphan_customers:,}")
print(f"   NULL product_sk  : {orphan_products:,}")
print(f"   NULL store_sk    : {orphan_stores:,}")
print(f"   NULL date_sk     : {orphan_dates:,}")

In [0]:
df_fact_enriched = (
    df_fact
    .withColumn("order_year", year(col("order_date")))
    .withColumn("profit_margin",
        spark_round(
            when(col("revenue") > 0, col("profit") / col("revenue"))
           .otherwise(lit(0.0)), 4
        )
    )
    .withColumn("discount_amount",
        spark_round(col("unit_price") * col("quantity") * col("discount"), 2)
    )
    .withColumn("gross_sales",
        spark_round(col("unit_price") * col("quantity"), 2)
    )
    .withColumn("ingestion_time", current_timestamp())
    .withColumn("updated_at",     current_timestamp())
)

In [0]:
df_final = df_fact_enriched.select(
    col("order_id"),
    col("order_date"),
    col("order_year"),
    col("date_sk"),
    col("customer_sk"),
    col("product_sk"),
    col("store_sk"),
    col("quantity"),
    col("unit_price"),
    col("discount"),
    col("discount_amount"),
    col("gross_sales"),
    col("revenue"),
    col("cost"),
    col("profit"),
    col("profit_margin"),
    col("is_valid"),
    col("ingestion_time"),
    col("source_file"),
    col("updated_at"),
)

print("Final fact schema:")
df_final.printSchema()
df_final.show(5, truncate=False)

In [0]:
if not spark.catalog.tableExists(f"{SILVER_DB}.{TARGET_TABLE}"):
    row_count = df_final.count()
    num_partitions = max(2, math.ceil(row_count / 10000))
    print(f"{row_count:,} rows -> {num_partitions} partitions")

    df_final.repartition(num_partitions).write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"{SILVER_DB}.{TARGET_TABLE}")
    print(f"{SILVER_DB}.{TARGET_TABLE} created")

else:
    df_final.createOrReplaceTempView("fact_sales_updates")

    spark.sql(f"""
        MERGE INTO {SILVER_DB}.{TARGET_TABLE}  AS target
        USING fact_sales_updates               AS source
        ON target.order_id = source.order_id

        WHEN MATCHED THEN UPDATE SET
            target.date_sk          = source.date_sk,
            target.customer_sk      = source.customer_sk,
            target.product_sk       = source.product_sk,
            target.store_sk         = source.store_sk,
            target.quantity         = source.quantity,
            target.unit_price       = source.unit_price,
            target.discount         = source.discount,
            target.discount_amount  = source.discount_amount,
            target.gross_sales      = source.gross_sales,
            target.revenue          = source.revenue,
            target.cost             = source.cost,
            target.profit           = source.profit,
            target.profit_margin    = source.profit_margin,
            target.is_valid         = source.is_valid,
            target.updated_at       = source.updated_at

        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"MERGE complete -> {SILVER_DB}.{TARGET_TABLE}")

In [0]:
print("Row count")
spark.sql(f"SELECT COUNT(*) AS total_orders FROM {SILVER_DB}.{TARGET_TABLE}").show()

print("Revenue by year")
spark.sql(f"""
    SELECT
        order_year,
        COUNT(*)                        AS orders,
        ROUND(SUM(revenue),  2)         AS total_revenue,
        ROUND(SUM(profit),   2)         AS total_profit,
        ROUND(AVG(profit_margin)*100,2) AS avg_margin_pct
    FROM {SILVER_DB}.{TARGET_TABLE}
    WHERE is_valid = 1
    GROUP BY order_year
    ORDER BY order_year
""").show()

print("Star schema join test: revenue by brand & country")
spark.sql(f"""
    SELECT
        p.brand,
        s.country,
        COUNT(f.order_id)          AS orders,
        ROUND(SUM(f.revenue), 2)   AS revenue,
        ROUND(SUM(f.profit),  2)   AS profit
    FROM {SILVER_DB}.fact_sales     f
    JOIN {SILVER_DB}.dim_products   p ON f.product_sk  = p.product_sk
    JOIN {SILVER_DB}.dim_stores     s ON f.store_sk    = s.store_sk
    WHERE f.is_valid = 1
    GROUP BY p.brand, s.country
    ORDER BY revenue DESC
    LIMIT 15
""").show()

print("Star schema join test: monthly revenue trend")
spark.sql(f"""
    SELECT
        c.year_month,
        c.month_name,
        COUNT(f.order_id)         AS orders,
        ROUND(SUM(f.revenue), 2)  AS revenue
    FROM {SILVER_DB}.fact_sales    f
    JOIN {SILVER_DB}.dim_calendar  c ON f.date_sk = c.date_sk
    WHERE f.is_valid = 1
    GROUP BY c.year_month, c.month_name
    ORDER BY c.year_month
""").show(24)